# Semana 8: análises exploratórias E1, E2, E4 e E6

Roda `analysis/exploratoria/exploratorias.py`, que segue `prereg/plano_exploratorio.md`. **Tudo aqui é exploratório**: nenhuma categoria do pré-registro é aplicada e nada substitui as tabelas da rodada confirmatória.

- **E1.** Gráfico por evento de Δ_shift (R3) e Δ_reg, com a parcela de pixels de cada evento.
- **E2.** Erros não sinalizados por classe (falsa água contra falsa terra).
- **E4.** Conjunto dos 5 seeds e variância entre seeds, por evento; conjunto R3 menos conjunto R1 com intervalo de 90%.
- **E6.** Δ_reg por unidade contra a fração de água, dentro de cada evento (só gera hipótese).

E3 (distância à borda) e E5 (R0 pelo pipeline da semana 6) ficam em notebooks próprios.

Antes de calcular, o script confere que lê as margens como a rodada confirmatória: τ de cada execução, AUROC_e por evento, erro não sinalizado e os valores agregados do conjunto têm de bater com `resultados/semana6/analise/resultados.json`. Se algo diferir, ele para.

| Célula | O que faz |
|---|---|
| 1 | Monta o Drive e confere RAM, disco e pastas |
| 2 | Clona o repositório, confere os scripts congelados e mostra a data do commit do plano |
| 3 | Copia as margens para o disco local e confere cada arquivo pelo manifesto (pula o que já foi copiado) |
| 4 | Roda as análises (cerca de 1 hora; só CPU, não precisa de GPU) |
| 5 | Copia a saída e o log para `resultados/semana8/exploratorias` |

Use um ambiente com pelo menos 8 GB de RAM. Se alguma célula der erro, pare e me mande a saída inteira.

Depois da célula 5, me mande os arquivos de `resultados/semana8/exploratorias`: a pasta `tabelas`, a pasta `figuras` e `exploratorias.log`.

In [ ]:
import os, re, json, time, hashlib, shutil, subprocess, sys, csv
import psutil

from google.colab import drive
drive.mount('/content/drive')
RAIZ = '/content/drive/MyDrive/Mestrado'
assert os.path.isdir(f'{RAIZ}/modelo_fixado'), 'Pasta Mestrado/modelo_fixado não encontrada. Confira se o Colab está na conta do mestrado.'

MARGENS_DRIVE = f'{RAIZ}/resultados/semana6/margens'
CHAVE = f'{RAIZ}/resultados/semana6_chave/chave_codigos.json'
CONFIRMATORIA = f'{RAIZ}/resultados/semana6/analise'
SAIDA_DRIVE = f'{RAIZ}/resultados/semana8/exploratorias'
assert os.path.isdir(MARGENS_DRIVE), 'Pasta resultados/semana6/margens não encontrada.'
assert os.path.isfile(CHAVE), 'A chave dos códigos não foi encontrada.'
assert os.path.isfile(f'{CONFIRMATORIA}/resultados.json'), 'resultados.json da rodada confirmatória não encontrado.'
assert not (os.path.isdir(SAIDA_DRIVE) and os.listdir(SAIDA_DRIVE)), (
    'A pasta resultados/semana8/exploratorias já tem arquivos. Não sobrescrevo: me avise antes de rodar de novo.')

def sha256_arquivo(caminho):
    h = hashlib.sha256()
    with open(caminho, 'rb') as f:
        for bloco in iter(lambda: f.read(1 << 24), b''):
            h.update(bloco)
    return h.hexdigest()

ram = psutil.virtual_memory().total / 1e9
disco = shutil.disk_usage('/content')[2] / 1e9
print(f'RAM da sessão: {ram:.1f} GB (as análises usam cerca de 4 GB no pico). Disco local livre: {disco:.0f} GB.')
assert ram >= 8, 'RAM abaixo de 8 GB. Use um ambiente com mais memória.'
assert disco > 30, 'Disco local com menos de 30 GB livres.'

In [ ]:
REPO_URL = 'https://github.com/RoseBorges44/prithvi-confidence-shift'
REPO = '/content/repo'
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', '-q', REPO_URL, REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=True)
commit = subprocess.run(['git', '-C', REPO, 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()

# os quatro arquivos do script confirmatório têm de ser os congelados (o script exploratório confere de novo)
tabela = open(f'{REPO}/analysis/README.md', encoding='utf-8').read()
registrados = dict(re.findall(r'\| `([a-z_]+\.py)` \| `([0-9a-f]{64})` \|', tabela))
assert set(registrados) == {'analise_congelada.py', 'nucleo.py', 'reamostragem.py', 'categorias.py'}, registrados.keys()
for nome, sha in registrados.items():
    assert sha256_arquivo(f'{REPO}/analysis/{nome}') == sha, f'{nome} é diferente do congelado. Não rode.'

data_plano = subprocess.run(['git', '-C', REPO, 'log', '--diff-filter=A', '-1', '--format=%cI %h', '--',
                             'prereg/plano_exploratorio.md'], capture_output=True, text=True, check=True).stdout.strip()
assert data_plano, 'prereg/plano_exploratorio.md não está no repositório.'
SHA_EXPLORATORIAS = sha256_arquivo(f'{REPO}/analysis/exploratoria/exploratorias.py')
print(f'Repositório no commit {commit[:7]}. Scripts congelados conferem.')
print(f'Plano exploratório adicionado em {data_plano}.')

In [ ]:
LOCAL = '/content/margens'
os.makedirs(LOCAL, exist_ok=True)
manifesto = list(csv.DictReader(open(f'{MARGENS_DRIVE}/manifesto_sha256.csv', newline='')))
assert len(manifesto) == 64, f'O manifesto tem {len(manifesto)} arquivos, esperado 64 (60 execuções e 4 rótulos).'
t0 = time.time()
for linha in manifesto:
    destino = f'{LOCAL}/{linha["arquivo"]}'
    if not (os.path.exists(destino) and os.path.getsize(destino) == int(linha['bytes'])):
        shutil.copy(f'{MARGENS_DRIVE}/{linha["arquivo"]}', destino + '.parcial')
        os.replace(destino + '.parcial', destino)
    assert sha256_arquivo(destino) == linha['sha256'], f'{linha["arquivo"]} diferente do manifesto.'
print(f'{len(manifesto)} arquivos copiados e conferidos pelo manifesto em {(time.time() - t0) / 60:.1f} min.')

In [ ]:
# Análises exploratórias (cerca de 1 hora). Deixe a aba aberta.
SAIDA = '/content/exploratorias'
LOG = '/content/exploratorias.log'
if os.path.isdir(SAIDA):
    shutil.rmtree(SAIDA)   # saída local de uma tentativa interrompida; a do Drive não é tocada
cmd = (f'cd {REPO} && set -o pipefail && python analysis/exploratoria/exploratorias.py --repo . --margens {LOCAL} '
       f'--chave "{CHAVE}" --confirmatoria "{CONFIRMATORIA}" --saida {SAIDA} 2>&1 | tee {LOG}')
get_ipython().system(f"bash -c '{cmd}'")

In [ ]:
assert os.path.isfile(f'{SAIDA}/exploratorias.json'), 'As análises não terminaram: não há exploratorias.json. Guarde o log e me mande.'
os.makedirs(SAIDA_DRIVE, exist_ok=True)
shutil.copytree(SAIDA, SAIDA_DRIVE, dirs_exist_ok=True)
shutil.copy(LOG, f'{SAIDA_DRIVE}/exploratorias.log')
INFO = {'commit_repositorio': commit, 'plano': 'prereg/plano_exploratorio.md', 'plano_adicionado': data_plano,
        'exploratorias_sha256': SHA_EXPLORATORIAS, 'congelados_sha256': registrados,
        'comando': 'analysis/exploratoria/exploratorias.py', 'gerado_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}
json.dump(INFO, open(f'{SAIDA_DRIVE}/rodada.json', 'w'), indent=2)
for raiz, _, arquivos in os.walk(SAIDA_DRIVE):
    for a in sorted(arquivos):
        print(os.path.relpath(os.path.join(raiz, a), SAIDA_DRIVE))
print(f'\nSaída copiada para {SAIDA_DRIVE}.')